# `mboostrs` — scikit-learn integration

`mboostrs` provides estimators that follow the scikit-learn API:

| Estimator | Description |
|---|---|
| `GLMBoostRegressor` | Linear gradient boosting regressor |
| `GLMBoostClassifier` | Linear gradient boosting classifier |
| `GAMBoostRegressor` | Additive model regressor (smooth terms) |
| `GAMBoostClassifier` | Additive model classifier |

This means you get:
- `fit()` / `predict()` / `score()` — the standard sklearn interface
- `cross_val_score()` — sklearn's cross-validation machinery
- `GridSearchCV()` — hyperparameter tuning
- `Pipeline` and `ColumnTransformer` — preprocessing integration
- `.get_params()` / `.set_params()` / `clone()` — estimator protocol

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import mboostrs as mb
import pandas as pd
import altair as alt
import polars as pl
from mboostrs.sklearn import (
    GLMBoostRegressor,
    GAMBoostRegressor,
    GLMBoostClassifier,
    GAMBoostClassifier,
)
from mboostrs.datasets import bodyfat, iris, wpbc

rng = np.random.default_rng(42)
print("mboostrs version:", mb.__version__)


## 1. GLMBoostRegressor — basic regression

`GLMBoostRegressor` wraps `glmboost` in sklearn's `fit(X, y)` interface. It expects a 2D NumPy array (or DataFrame) `X` and a 1D array `y`.

The data has $n=500$ observations, $p=10$ predictors, of which only **4** are truly relevant (true coefficients: 1.5, -0.8, 0.6, -0.3, rest zero).

In [ ]:
n, p = 500, 10
X = rng.normal(size=(n, p))
true_beta = np.array([1.5, -0.8, 0, 0.6, 0, 0, 0, -0.3, 0, 0])
y = X @ true_beta + rng.normal(0, 0.3, n)

reg = GLMBoostRegressor(mstop=200, nu=0.1)
reg.fit(X, y)
print(f"R^2 score: {reg.score(X, y):.4f}")
print(f"Non-zero coefficients: {sum(1 for v in reg.model_.coefficients_.values() if abs(v) > 0.01)}")

In [ ]:
# Predict vs actual scatter
y_pred = reg.predict(X)
pred_df = pd.DataFrame({"actual": y, "predicted": y_pred})
lim = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
alt.layer(
    alt.Chart(pred_df).mark_point(opacity=0.3).encode(
        x="actual", y="predicted"
    ),
    alt.Chart(pd.DataFrame({"x": lim, "y": lim})).mark_line(
        color="red", strokeDash=[4, 4]
    ).encode(x="x", y="y"),
).properties(title=f"GLMBoostRegressor \u2014 R\u00b2 = {reg.score(X, y):.4f}")


## 2. GAMBoostRegressor — smooth terms via `bbs`

`GAMBoostRegressor` wraps `gamboost` with `bbs` smooths. By default, all numeric predictors get a `bbs(..., df=4)` term. You can also pass a custom formula for fine-grained control.

**Data:** $y = \sin(2\pi x_0) + 0.5 \cdot x_1 + \varepsilon$ — nonlinear in $x_0$, linear in $x_1$.

In [ ]:
n = 300
X_gam = np.column_stack([
    rng.uniform(0, 1, n),
    rng.normal(size=n),
])
y_gam = np.sin(2 * np.pi * X_gam[:, 0]) + 0.5 * X_gam[:, 1] + rng.normal(0, 0.2, n)

# Default: all predictors get bbs(df=4)
gam = GAMBoostRegressor(df=4, mstop=200, nu=0.1)
gam.fit(X_gam, y_gam)
print(f"GAM R^2 (default bbs terms): {gam.score(X_gam, y_gam):.4f}")

# Custom formula: bbs for f0 (nonlinear), bols for f1 (linear)
gam_f = GAMBoostRegressor(
    formula="y ~ bbs(f0, df=5) + bols(f1)",
    mstop=200, nu=0.1,
)
gam_f.fit(X_gam, y_gam)
print(f"GAM R^2 (custom formula): {gam_f.score(X_gam, y_gam):.4f}")

## 3. GLMBoostClassifier — binary classification

`GLMBoostClassifier` wraps `glmboost` with `Binomial` family. It provides both `predict()` (hard classes) and `predict_proba()` (calibrated probabilities).

**Data:** $\eta = x_0 - 0.6 \cdot x_1 + 0.3 \cdot x_2$ with logistic link, $y \sim \text{Bernoulli}(p)$.

In [ ]:
n = 400
X_clf = rng.normal(size=(n, 5))
eta = X_clf[:, 0] - 0.6 * X_clf[:, 1] + 0.3 * X_clf[:, 2]
p = 1 / (1 + np.exp(-eta))
y_clf = (rng.uniform(size=n) < p).astype(int)

clf = GLMBoostClassifier(mstop=200, nu=0.1)
clf.fit(X_clf, y_clf)
print(f"Accuracy: {clf.score(X_clf, y_clf):.3f}")

# Predicted probabilities (well-calibrated thanks to logistic loss)
probs = clf.predict_proba(X_clf[:5])
print(f"Probabilities (first 5 rows):")
print(probs)

## 4. GAMBoostClassifier — smooth classifier

`GAMBoostClassifier` uses smooth `bbs` terms for non-linear decision boundaries in classification.

In [ ]:
n = 400
X_gc = np.column_stack([
    rng.uniform(-2, 2, n),
    rng.normal(size=n),
])
eta_gc = np.sin(X_gc[:, 0]) + 0.5 * X_gc[:, 1]
p_gc = 1 / (1 + np.exp(-eta_gc))
y_gc = (rng.uniform(size=n) < p_gc).astype(int)

gc = GAMBoostClassifier(df=4, mstop=200, nu=0.1)
gc.fit(X_gc, y_gc)
print(f"GAM classifier accuracy: {gc.score(X_gc, y_gc):.3f}")

## 5. Cross-validation with sklearn

Because the wrappers follow sklearn conventions, `cross_val_score` works directly — no mboostrs-specific CV API needed.

In [ ]:
from sklearn.model_selection import cross_val_score

n, p = 300, 8
X_cv = rng.normal(size=(n, p))
y_cv = X_cv[:, 0] + 0.5 * X_cv[:, 1] + rng.normal(0, 0.3, n)

glm_cv = GLMBoostRegressor(mstop=100, nu=0.1)
scores = cross_val_score(glm_cv, X_cv, y_cv, cv=5, scoring="r2")
print(f"CV R^2 scores: {scores}")
print(f"Mean R^2: {scores.mean():.4f}")

gam_cv = GAMBoostRegressor(df=4, mstop=100, nu=0.1)
scores_gam = cross_val_score(gam_cv, X_cv, y_cv, cv=5, scoring="r2")
print(f"GAM CV R^2: {scores_gam.mean():.4f}")

In [ ]:
# CV scores comparison
cv_df = pd.DataFrame({
    "model": ["GLM"] * len(scores) + ["GAM"] * len(scores_gam),
    "R\u00b2": np.concatenate([scores, scores_gam]),
})

alt.Chart(cv_df).mark_boxplot(extent="min-max").encode(
    x="model:N", y="R\u00b2:Q", color="model:N",
).properties(title="5-fold CV R\u00b2: GLM vs GAM")


## 6. GridSearchCV

`GridSearchCV` evaluates the specified `mstop` and `nu` values by five-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

n, p = 400, 10
X_gs = rng.normal(size=(n, p))
y_gs = X_gs[:, 0] - 0.5 * X_gs[:, 1] + 0.3 * X_gs[:, 5] + rng.normal(0, 0.2, n)

param_grid = {
    "mstop": [50, 100, 200, 400],
    "nu": [0.05, 0.1, 0.2],
}
grid = GridSearchCV(
    GLMBoostRegressor(),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=1,
)
grid.fit(X_gs, y_gs)
print(f"Best params: {grid.best_params_}")
print(f"Best CV MSE: {-grid.best_score_:.4f}")

In [ ]:
# Grid search results
gs_df = pd.DataFrame(grid.cv_results_)
pivot = gs_df.pivot_table(index="param_nu", columns="param_mstop",
                          values="mean_test_score")
pivot.index.name = "nu"
pivot.columns.name = "mstop"
pivot = pivot.reset_index().melt(id_vars="nu", var_name="mstop", value_name="neg_MSE")

alt.Chart(pivot).mark_rect().encode(
    x=alt.X("mstop:O", title="mstop"),
    y=alt.Y("nu:O", title="\u03bd"),
    color=alt.Color("neg_MSE:Q", scale=alt.Scale(scheme="viridis")),
    tooltip=["nu", "mstop", "neg_MSE"],
).properties(title="GridSearchCV \u2014 negative MSE")


## 7. Pipeline with ColumnTransformer

Real workflows need preprocessing: scaling, encoding, imputation. The sklearn wrappers integrate with `Pipeline` and `ColumnTransformer`.

Here we scale numeric features before passing them to the boosted model.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

import pandas as pd

n = 200
age = rng.uniform(18, 80, n)
income = rng.exponential(50, n)
education = rng.choice([0, 1, 2, 3], size=n)

y_pipe = 0.02 * age + 0.005 * income + 0.2 * education + rng.normal(0, 0.2, n)

df = pd.DataFrame({
    "age": age, "income": income, "education": education,
})

ct = ColumnTransformer([
    ("num", StandardScaler(), ["age", "income"]),
])

pipe = Pipeline([
    ("preprocess", ct),
    ("gb", GLMBoostRegressor(mstop=200, nu=0.1)),
])
pipe.fit(df, y_pipe)
print(f"Pipeline score: {pipe.score(df, y_pipe):.4f}")

## 8. Real datasets

Testing on real data: bodyfat regression and Iris classification.

In [ ]:
# Bodyfat regression
bf = bodyfat()
y_bf = bf["DEXfat"].to_numpy()
X_bf = bf.drop("DEXfat").to_numpy()

reg_bf = GLMBoostRegressor(mstop=300, nu=0.1)
reg_bf.fit(X_bf, y_bf)
print(f"Bodyfat R^2: {reg_bf.score(X_bf, y_bf):.4f}")

# Iris setosa vs rest (binary via GAMBoostClassifier)
iris_df = iris()
X_ir = iris_df[["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"]].to_numpy()
y_ir = (iris_df["Species"] == "setosa").to_numpy().astype(int)

clf_ir = GAMBoostClassifier(df=4, mstop=100, nu=0.1)
clf_ir.fit(X_ir, y_ir)
print(f"Iris setosa classifier accuracy: {clf_ir.score(X_ir, y_ir):.3f}")

## 9. Estimator protocol: `clone`, `get_params`, `set_params`

Full sklearn estimator protocol support — essential for meta-estimators.

In [ ]:
from sklearn.base import clone

reg_a = GLMBoostRegressor(mstop=100, nu=0.1)
reg_b = clone(reg_a)
print(f"Clone params: {reg_b.get_params()}")

reg_b.set_params(mstop=200, nu=0.05)
print(f"Modified: mstop={reg_b.mstop}, nu={reg_b.nu}")

## 10. Weighted regression

Observation weights are supported (passed through to the underlying `glmboost` / `gamboost` fit). This is useful for survey sampling, cost-sensitive learning, or down-weighting low-quality observations.

In [ ]:
n = 300
X_w = rng.normal(size=(n, 3))
y_w = X_w[:, 0] + 0.5 * X_w[:, 1] + rng.normal(0, 0.2, n)

reg_w = GLMBoostRegressor(mstop=100, nu=0.1)
reg_w.fit(X_w, y_w)
print(f"Weighted-compatible fit: R^2={reg_w.score(X_w, y_w):.4f}")